In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the quality check summary

summary_path = "../output/reports/quality_check_summary.csv"

quality_summary = pd.read_csv(summary_path)

quality_summary

,Quality Check,Issue Count,Issue Percentage,Status
0,Missing Values,0,0.0,Passed
1,Empty Text Values,0,0.0,Passed
2,Duplicate IDs,0,0.0,Passed
3,Duplicate SFT Samples,9980,99.8,Major Issue
4,Very Short Instructions,0,0.0,Passed
5,Very Short Inputs,0,0.0,Passed
6,Very Short Outputs,0,0.0,Passed
7,Very Long Outputs,0,0.0,Passed
8,Empty Domains,0,0.0,Passed
9,Domains With Extra Spaces,0,0.0,Passed


In [5]:
df = pd.read_excel("C:\\Users\\Legion\\Downloads\\SFT_Contextual_10000_updated.xlsx")
print("Dataset loaded successfully!")
print("Total samples:", len(df))
df

Dataset loaded successfully!
Total samples: 10000


,id,domain,instruction,input,output,source
0,8bcd7516-264f-4d52-8d24-223a4519d71b,finance,Calculate compound interest from the details b...,"Principal: $1,000, Rate: 5%, Time: 3 years",Using the compound interest formula A = P(1 + ...,synthetic
1,02b6b474-7e0a-4d26-b58d-26caeda87200,marketing,Suggest a campaign slogan from this product de...,A smartwatch that monitors your health in real...,"""Your Health, Live. Your Performance, Elevated...",human-curated
2,13741ac3-de4c-4f18-ad04-0f9aca9e8434,productivity,Create a daily schedule using this task list.,"Write report, team meeting, client call, gym, ...",Here is a suggested daily schedule:\n8:00–9:30...,synthetic
3,01de4e88-64aa-45f3-befb-c077e83e5418,sports,Summarize the game report.,Team A scored 3 goals in the second half to wi...,"Team A came from behind, scoring all 3 of thei...",synthetic
4,b04b6a43-d366-4b48-bbb7-9f2e075205e2,law,Summarize this legal paragraph.,The Fourth Amendment protects citizens against...,The Fourth Amendment safeguards individuals fr...,human-curated
...,...,...,...,...,...,...
9995,28b08e2c-1491-4bcb-9081-f2d9c3edd813,healthcare,Summarize the patient's condition from the not...,"Patient reports shortness of breath, fatigue, ...",The patient has presented with a 3-day history...,human-curated
9996,c438da14-6945-46d4-b5e1-109cbdf6365d,education,Summarize the paragraph below.,The Industrial Revolution marked a major turni...,The Industrial Revolution was a pivotal period...,human-curated
9997,93e27108-959a-40cb-abec-0a03b0ea07fa,sports,Summarize the game report.,Team A scored 3 goals in the second half to wi...,"Team A came from behind, scoring all 3 of thei...",human-curated
9998,0cc166fa-7b81-407c-bbf5-57df5fec39a7,law,Explain if the scenario below qualifies as neg...,A store employee mopped the floor but failed t...,This likely qualifies as negligence. The store...,human-curated


In [19]:
# Create a dictionary of quality issue counts

issue_counts = dict(
    zip(
        quality_summary["Quality Check"],
        quality_summary["Issue Count"]
    )
)

issue_counts

{'Missing Values': 0,
 'Empty Text Values': 0,
 'Duplicate IDs': 0,
 'Duplicate SFT Samples': 9980,
 'Very Short Instructions': 0,
 'Very Short Inputs': 0,
 'Very Short Outputs': 0,
 'Very Long Outputs': 0,
 'Empty Domains': 0,
 'Domains With Extra Spaces': 0,
 'Empty Sources': 0,
 'Sources With Extra Spaces': 0}

In [20]:
# Calculate duplicate SFT samples

duplicate_sft_count = (
    df.duplicated(
        subset=[
            "instruction",
            "input",
            "output"
        ]
    )
    .sum()
)

print(
    "Duplicate SFT samples:",
    duplicate_sft_count
)

Duplicate SFT samples: 9980


In [21]:
# Calculate the duplicate rate

total_samples = len(df)

duplicate_rate = (
    duplicate_sft_count
    / total_samples
    * 100
)

duplicate_rate = round(
    duplicate_rate,
    2
)

print(
    "Duplicate rate:",
    duplicate_rate,
    "%"
)

Duplicate rate: 99.8 %


In [22]:
# Assign a penalty based on duplicate rate

def get_duplicate_penalty(
    duplicate_rate
):

    if duplicate_rate == 0:

        return 0

    elif duplicate_rate <= 5:

        return 5

    elif duplicate_rate <= 20:

        return 15

    elif duplicate_rate <= 50:

        return 30

    else:

        return 50

In [23]:
# Calculate the duplicate penalty

duplicate_penalty = (
    get_duplicate_penalty(
        duplicate_rate
    )
)

print(
    "Duplicate penalty:",
    duplicate_penalty
)

Duplicate penalty: 50


In [24]:
# Define penalties for other quality issues

other_penalty_weights = {
    "Missing Values": 20,
    "Empty Text Values": 15,
    "Duplicate IDs": 10,
    "Very Short Instructions": 5,
    "Very Short Inputs": 5,
    "Very Short Outputs": 10,
    "Very Long Outputs": 5,
    "Empty Domains": 5,
    "Domains With Extra Spaces": 1,
    "Empty Sources": 5,
    "Sources With Extra Spaces": 1
}

In [25]:
# Calculate penalty for other quality issues

def calculate_issue_penalty(
    issue_count,
    total_samples,
    maximum_penalty
):

    issue_rate = (
        issue_count
        / total_samples
    )

    penalty = (
        issue_rate
        * maximum_penalty
    )

    return penalty

In [26]:
# Calculate penalties for other quality issues

penalty_results = []

for issue_name, maximum_penalty in (
    other_penalty_weights.items()
):

    issue_count = (
        issue_counts.get(
            issue_name,
            0
        )
    )

    penalty = (
        calculate_issue_penalty(
            issue_count,
            total_samples,
            maximum_penalty
        )
    )

    penalty_results.append(
        {
            "Quality Check": issue_name,
            "Issue Count": issue_count,
            "Maximum Penalty": maximum_penalty,
            "Penalty Applied": round(
                penalty,
                2
            )
        }
    )

penalty_summary = pd.DataFrame(
    penalty_results
)

penalty_summary

,Quality Check,Issue Count,Maximum Penalty,Penalty Applied
0,Missing Values,0,20,0.0
1,Empty Text Values,0,15,0.0
2,Duplicate IDs,0,10,0.0
3,Very Short Instructions,0,5,0.0
4,Very Short Inputs,0,5,0.0
5,Very Short Outputs,0,10,0.0
6,Very Long Outputs,0,5,0.0
7,Empty Domains,0,5,0.0
8,Domains With Extra Spaces,0,1,0.0
9,Empty Sources,0,5,0.0


In [27]:
# Calculate total penalty from other issues

other_total_penalty = (
    penalty_summary[
        "Penalty Applied"
    ]
    .sum()
)

other_total_penalty = round(
    other_total_penalty,
    2
)

print(
    "Other quality issue penalty:",
    other_total_penalty
)

Other quality issue penalty: 0.0


In [28]:
# Calculate the final quality score

starting_score = 100

total_penalty = (
    duplicate_penalty
    + other_total_penalty
)

final_quality_score = (
    starting_score
    - total_penalty
)

final_quality_score = max(
    0,
    round(
        final_quality_score,
        2
    )
)

print(
    "Starting score:",
    starting_score
)

print(
    "Duplicate penalty:",
    duplicate_penalty
)

print(
    "Other penalties:",
    other_total_penalty
)

print(
    "Total penalty:",
    total_penalty
)

print(
    "Final quality score:",
    final_quality_score
)

Starting score: 100
Duplicate penalty: 50
Other penalties: 0.0
Total penalty: 50.0
Final quality score: 50.0


In [29]:
# Assign a quality status

def get_quality_status(
    quality_score
):

    if quality_score >= 90:

        return "Excellent"

    elif quality_score >= 75:

        return "Good"

    elif quality_score >= 60:

        return "Fair"

    elif quality_score >= 40:

        return "Poor"

    else:

        return "Critical"

In [30]:
# Get the final quality status

quality_status = (
    get_quality_status(
        final_quality_score
    )
)

print(
    "Quality status:",
    quality_status
)

Quality status: Poor


In [31]:
# Add duplicate information to the penalty report

duplicate_penalty_row = pd.DataFrame({
    "Quality Check": [
        "Duplicate SFT Samples"
    ],
    "Issue Count": [
        duplicate_sft_count
    ],
    "Maximum Penalty": [
        50
    ],
    "Penalty Applied": [
        duplicate_penalty
    ]
})

complete_penalty_summary = pd.concat(
    [
        duplicate_penalty_row,
        penalty_summary
    ],
    ignore_index=True
)

complete_penalty_summary

,Quality Check,Issue Count,Maximum Penalty,Penalty Applied
0,Duplicate SFT Samples,9980,50,50.0
1,Missing Values,0,20,0.0
2,Empty Text Values,0,15,0.0
3,Duplicate IDs,0,10,0.0
4,Very Short Instructions,0,5,0.0
5,Very Short Inputs,0,5,0.0
6,Very Short Outputs,0,10,0.0
7,Very Long Outputs,0,5,0.0
8,Empty Domains,0,5,0.0
9,Domains With Extra Spaces,0,1,0.0


In [33]:
# Create the final quality scoring result

scoring_result = pd.DataFrame({
    "Metric": [
        "Total Samples",
        "Duplicate SFT Samples",
        "Duplicate Rate",
        "Starting Score",
        "Duplicate Penalty",
        "Other Penalties",
        "Total Penalty",
        "Final Quality Score",
        "Quality Status"
    ],
    "Value": [
        total_samples,
        duplicate_sft_count,
        f"{duplicate_rate}%",
        starting_score,
        duplicate_penalty,
        other_total_penalty,
        total_penalty,
        f"{final_quality_score}%",
        quality_status
    ]
})

scoring_result

,Metric,Value
0,Total Samples,10000
1,Duplicate SFT Samples,9980
2,Duplicate Rate,99.8%
3,Starting Score,100
4,Duplicate Penalty,50
5,Other Penalties,0.0
6,Total Penalty,50.0
7,Final Quality Score,50.0%
8,Quality Status,Poor


In [34]:
# Save the final scoring result

output_path = (
    "../output/reports/"
    "quality_scoring_result.csv"
)

scoring_result.to_csv(
    output_path,
    index=False
)

print(
    "Quality scoring result saved successfully"
)

Quality scoring result saved successfully


In [35]:
# Save the detailed penalty report

penalty_path = (
    "../output/reports/"
    "quality_penalty_details.csv"
)

complete_penalty_summary.to_csv(
    penalty_path,
    index=False
)

print(
    "Quality penalty details saved successfully"
)

Quality penalty details saved successfully


In [36]:
# Display the final quality scoring report

print("SFT DATASET QUALITY SCORING REPORT")

print()

print(
    "Total samples:",
    total_samples
)

print(
    "Duplicate SFT samples:",
    duplicate_sft_count
)

print(
    "Duplicate rate:",
    f"{duplicate_rate}%"
)

print(
    "Starting score:",
    starting_score
)

print(
    "Duplicate penalty:",
    duplicate_penalty
)

print(
    "Other penalties:",
    other_total_penalty
)

print(
    "Total penalty:",
    total_penalty
)

print(
    "Final quality score:",
    f"{final_quality_score}/100"
)

print(
    "Quality status:",
    quality_status
)

SFT DATASET QUALITY SCORING REPORT

Total samples: 10000
Duplicate SFT samples: 9980
Duplicate rate: 99.8%
Starting score: 100
Duplicate penalty: 50
Other penalties: 0.0
Total penalty: 50.0
Final quality score: 50.0/100
Quality status: Poor
